### Welcome to our Big Data Project 
Members :
* Mohamed Ibrahim Said Abo Salim
* Mohamed Nasser 
* Ahmed Hamoda
* Abdalrahman Othman


# US Accidents Dataset (December 2021 , March 2023)

The US Accidents dataset is a large-scale, real-world dataset containing traffic accident records across the United States. It is widely used in data science and machine learning research due to its high volume, diverse feature set, and strong spatial-temporal characteristics.

---

## Target Variable

### Severity

The target variable represents the severity level of a traffic accident. It is a categorical variable with values ranging from 1 to 4:

| Severity | Description |
|----------|------------|
| 1 | Minor accident with minimal impact |
| 2 | Moderate accident with limited disruption |
| 3 | Severe accident causing significant disruption |
| 4 | Very severe accident with critical impact |

This variable is used as the target for supervised machine learning classification tasks.

---

## Time-Related Features

The dataset includes several timestamp-based attributes:

- **Start_Time**: The time when the accident began  
- **End_Time**: The time when the accident ended  
- **Weather_Timestamp**: The time when weather conditions were recorded  

### Feature Engineering

From these features, additional temporal variables can be derived:

- Hour of the day  
- Day of the week  
- Month or season  
- Weekend versus weekday  

These derived features are important for analyzing temporal patterns in accidents.

---

## Location Features

The dataset contains detailed spatial and administrative location information:

- **Start_Lat, Start_Lng**: Latitude and longitude of the accident origin  
- **End_Lat, End_Lng**: Latitude and longitude of the accident end location  
- **City, County, State**: Administrative divisions  
- **Zipcode**: Postal code  
- **Country**: Country where the accident occurred  

### Usage

These features are used for:

- Geospatial analysis  
- Hotspot detection  
- Regional risk assessment  

---

## Road Condition Features

These are binary variables indicating the presence (1) or absence (0) of specific road elements:

- Amenity  
- Bump  
- Crossing  
- Give_Way  
- Junction  
- No_Exit  
- Railway  
- Roundabout  
- Station  
- Stop  
- Traffic_Signal  
- Traffic_Calming  
- Turning_Loop  

These variables provide information about road infrastructure and traffic control mechanisms.

---

## Weather Features

Weather conditions play a significant role in accident occurrence and severity. The dataset includes:

- **Temperature(F)**: Ambient temperature  
- **Wind_Chill(F)**: Perceived temperature due to wind  
- **Humidity(%)**: Atmospheric humidity  
- **Pressure(in)**: Atmospheric pressure  
- **Visibility(mi)**: Visibility distance  
- **Wind_Speed(mph)**: Wind speed  
- **Precipitation(in)**: Amount of precipitation  
- **Weather_Condition**: Categorical description of weather (e.g., clear, rain, fog, snow)  

These features are among the most important predictors of accident severity.

---

## Wind Feature

- **Wind_Direction**: Direction of wind (e.g., N, S, E, W)

This feature helps in understanding environmental driving conditions.

---

## Lighting and Environmental Conditions

- **Sunrise_Sunset**: Indicates whether the accident occurred during daytime or nighttime  
- **Civil_Twilight**  
- **Nautical_Twilight**  
- **Astronomical_Twilight**  

In most modeling scenarios, **Sunrise_Sunset** is commonly used as a simplified representation of lighting conditions.

---

## Additional Features

- **Airport_Code**: Identifier for the nearest airport weather station  
- **Timezone**: Time zone of the accident location  
- **Number**: Street number associated with the location (often sparse or missing)  

---

## Text Feature

### Description

A free-text field describing the accident event.

### Applications

- Text classification  
- Keyword extraction  
- Event summarization  
- Natural language processing tasks  

---

## Key Features for Modeling

The most significant variables for predicting accident severity include:

- Severity (target variable)  
- Start_Time (engineered into temporal features)  
- Weather_Condition  
- Visibility  
- Temperature  
- Traffic_Signal  
- Junction  
- Crossing  

---

## Summary

The dataset integrates multiple dimensions of information, including temporal, spatial, environmental, and infrastructural data. This makes it highly suitable for:

- Exploratory data analysis  
- Machine learning classification tasks  
- Pattern recognition  
- Predictive modeling of traffic accident severity  

installing all needed libraries for running the notebook using pip command

In [ ]:
%pip install pyspark
%pip install findspark

Importing all needed libraries , packages and modules

In [ ]:
import findspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, isnan, when, count, round as spark_round
from pyspark.sql.types import DoubleType, FloatType


Environment Setup

In [2]:
findspark.init()

Spark configuration to run the session

In [3]:
spark = SparkSession.builder \
    .appName("US Accidents Analysis - Local Laptop") \
    .master("local[*]") \
    .config("spark.driver.memory", "6g") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.default.parallelism", "16") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "512m") \
    .getOrCreate()

spark

The path to our *data* for the BigData project

In [4]:
DEC_file_path = "Data/US_Accidents_Dec21.csv"
MAR_file_path = "Data/US_Accidents_Mar23.csv"

Loading the datasets into Spark DataFrames

In [5]:
Dec_df = spark.read \
    .format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("mode", "DROPMALFORMED") \
    .option("multiLine", True) \
    .option("escape", '"') \
    .option("quote", '"') \
    .option("ignoreLeadingWhiteSpace", True) \
    .option("ignoreTrailingWhiteSpace", True) \
    .load(DEC_file_path)

In [6]:
MAR_df = spark.read \
    .format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("mode", "DROPMALFORMED") \
    .option("multiLine", True) \
    .option("escape", '"') \
    .option("quote", '"') \
    .option("ignoreLeadingWhiteSpace", True) \
    .option("ignoreTrailingWhiteSpace", True) \
    .load(MAR_file_path)

Repartition after loading

In [7]:
Dec_df = Dec_df.repartition(16)
MAR_df = MAR_df.repartition(16)

Printing the inferSchema for both of the data

In [12]:
Dec_df.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- End_Lat: double (nullable = true)
 |-- End_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Number: double (nullable = true)
 |-- Street: string (nullable = true)
 |-- Side: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(

In [13]:
MAR_df.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Source: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- End_Lat: double (nullable = true)
 |-- End_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nullable = true)
 |-- V

Checking of both of the inferSchema is the same to each others

In [16]:
cols1 = set(Dec_df.columns)
cols2 = set(MAR_df.columns)
common_cols = cols1.intersection(cols2)
print(f"Number of common columns: {len(common_cols)}")
diff_cols1 = cols1 - cols2
diff_cols2 = cols2 - cols1
print(f"Columns in Dec_df but not in MAR_df: {diff_cols1}")
print(f"Columns in MAR_df but not in Dec_df: {diff_cols2}")

Number of common columns: 45
Columns in Dec_df but not in MAR_df: {'Number', 'Side'}
Columns in MAR_df but not in Dec_df: {'Source'}


In [17]:
Dec_df.select("Number", "Side").show(5)

+------+----+
|Number|Side|
+------+----+
|1789.0|   R|
|1789.0|   R|
|1789.0|   R|
|2589.0|   L|
|2589.0|   L|
+------+----+
only showing top 5 rows


In [18]:
MAR_df.select("Source").show(5)

+-------+
| Source|
+-------+
|Source2|
|Source2|
|Source2|
|Source2|
|Source2|
+-------+
only showing top 5 rows


In [19]:
print("This is Dec_df:")
Dec_df.show(5)
print("This is MAR_df:")
MAR_df.show(5)

This is Dec_df:
+---------+--------+-------------------+-------------------+---------+-----------+------------------+------------------+-------------------+--------------------+------+---------------+----+---------------+--------------------+-----+----------+-------+----------+------------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+
|       ID|Severity|         Start_Time|           End_Time|Start_Lat|  Start_Lng|           End_Lat|           End_Lng|       Distance(mi)|         Description|Number|         Street|Side|           City|              County|State|   Zipcode|Country|  Timezone|Airport_Code|  Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibil

Showing how many Records in the both of the data also how many columns are there 

In [21]:
#making vars to carry the count of records and columns for both dataframes
dec_records = Dec_df.count()
mar_records = MAR_df.count()
dec_columns = len(Dec_df.columns)
mar_columns = len(MAR_df.columns)
print("Number of records in Dec_df:", dec_records)
print("Number of records in MAR_df:", mar_records)
print("Number of columns in Dec_df:", dec_columns)
print("Number of columns in MAR_df:", mar_columns)

Number of records in Dec_df: 2845342
Number of records in MAR_df: 7728394
Number of columns in Dec_df: 47
Number of columns in MAR_df: 46


In [24]:
print("Number of records in Both DataFrames:", dec_records + mar_records)


Number of records in Both DataFrames: 10573736


Showing statistical summary

In [23]:
print("The statistical summary of Dec_df:")
Dec_df.describe().show()
print("The statistical summary of MAR_df:")
MAR_df.describe().show()

The statistical summary of Dec_df:
+-------+--------+------------------+-----------------+------------------+-----------------+------------------+------------------+--------------------+------------------+--------------------+-------+----------+---------+-------+------------------+-------+----------+------------+------------------+------------------+-----------------+------------------+------------------+--------------+-----------------+--------------------+------------------+--------------+--------------+-----------------+---------------------+
|summary|      ID|          Severity|        Start_Lat|         Start_Lng|          End_Lat|           End_Lng|      Distance(mi)|         Description|            Number|              Street|   Side|      City|   County|  State|           Zipcode|Country|  Timezone|Airport_Code|    Temperature(F)|     Wind_Chill(F)|      Humidity(%)|      Pressure(in)|    Visibility(mi)|Wind_Direction|  Wind_Speed(mph)|   Precipitation(in)| Weather_Condition|Su

null/missing value analysis

In [32]:
def null_analysis(df, name):
    print(f"\n=== Null Analysis: {name} ===")
    total = df.count()
    
    null_exprs = []
    for field in df.schema.fields:
        c = field.name
        # isnan() is only valid for numeric (Double/Float) columns
        # For all other types, just check isNull()
        if isinstance(field.dataType, (DoubleType, FloatType)):
            condition = col(c).isNull() | isnan(c)
        else:
            condition = col(c).isNull()
        
        null_exprs.append(count(when(condition, c)).alias(c))
    
    null_counts = df.select(null_exprs)
    
    null_df = null_counts.toPandas().T.reset_index()
    null_df.columns = ["column", "null_count"]
    null_df["null_pct"] = (null_df["null_count"] / total * 100).round(2)
    null_df = null_df[null_df["null_count"] > 0].sort_values("null_pct", ascending=False)
    
    if null_df.empty:
        print("No nulls found.")
    else:
        print(null_df.to_string(index=False))

In [33]:
null_analysis(Dec_df, "Dec 2021")
null_analysis(MAR_df, "Mar 2023")


=== Null Analysis: Dec 2021 ===
               column  null_count  null_pct
               Number     1743911     61.29
    Precipitation(in)      549458     19.31
        Wind_Chill(F)      469643     16.51
      Wind_Speed(mph)      157944      5.55
       Wind_Direction       73775      2.59
          Humidity(%)       73092      2.57
       Visibility(mi)       70546      2.48
    Weather_Condition       70636      2.48
       Temperature(F)       69274      2.43
         Pressure(in)       59200      2.08
    Weather_Timestamp       50736      1.78
         Airport_Code        9549      0.34
             Timezone        3659      0.13
       Sunrise_Sunset        2867      0.10
    Nautical_Twilight        2867      0.10
       Civil_Twilight        2867      0.10
Astronomical_Twilight        2867      0.10
              Zipcode        1319      0.05
                 City         137      0.00
               Street           2      0.00

=== Null Analysis: Mar 2023 ===
          

In [34]:
total_dec = Dec_df.count()
Dec_df.groupBy("Severity") \
    .agg(count("*").alias("count")) \
    .withColumn("percentage", spark_round(col("count") / total_dec * 100, 2)) \
    .orderBy("Severity") \
    .show()

+--------+-------+----------+
|Severity|  count|percentage|
+--------+-------+----------+
|       1|  26053|      0.92|
|       2|2532991|     89.02|
|       3| 155105|      5.45|
|       4| 131193|      4.61|
+--------+-------+----------+



In [35]:
total_mar = MAR_df.count()
MAR_df.groupBy("Severity") \
    .agg(count("*").alias("count")) \
    .withColumn("percentage", spark_round(col("count") / total_mar * 100, 2)) \
    .orderBy("Severity") \
    .show()

+--------+-------+----------+
|Severity|  count|percentage|
+--------+-------+----------+
|       1|  67366|      0.87|
|       2|6156981|     79.67|
|       3|1299337|     16.81|
|       4| 204710|      2.65|
+--------+-------+----------+



In [8]:
total_duplicate_dec = Dec_df.count()
total_duplicate_MAR = MAR_df.count()
unique_dec = Dec_df.select("ID").distinct().count()
unique_MAR = MAR_df.select("ID").distinct().count()

print(f"Duplicate rows: {total_duplicate_dec - unique_dec} ({round((total_duplicate_dec-unique_dec)/total_duplicate_dec*100, 2)}%)")
print(f"Duplicate rows: {total_duplicate_MAR - unique_MAR} ({round((total_duplicate_MAR-unique_MAR)/total_duplicate_MAR*100, 2)}%)")

Duplicate rows: 0 (0.0%)
Duplicate rows: 0 (0.0%)
